# NB01 — 태스크 정의와 데이터셋 구축

> 논문 §3(태스크 정의), §4(데이터셋) 대응 노트북.

## 1. 태스크 정의: Conversational Affection Imbalance Detection

- **입력**: 2인 카카오톡 대화 $D = [(\text{화자}, \text{시각}, \text{메시지}, \text{메시지타입}), \dots]$
- **출력**: $\text{dominance} \in [0,1]$, $\text{dependence} \in [0,1]$ — 0.5가 균형, 1에 가까울수록 '나'(A)가 갑 / 더 의존적
- **gold label**: 합성 대화를 만든 **행동 파라미터에서 유도** (LLM 의견이 아님 → 순환성 방지)
- **평가지표** (`eval/metrics.py`): MAE, Spearman ρ, sign accuracy(갑/을 방향), ICC(2,1)·화자 스왑 대칭성(신뢰도)

## 2. 합성 프로토콜: script-then-verbalize

1. `build_spec(l_dom, l_dep, scenario)` — 불균형 수준 → 행동 파라미터(선톡 비율, 답장 지연, 발화량, 연속톡, 성의, 비언어 신호 빈도)
2. `build_turn_script(spec, seed)` — **LLM 없이** 결정적 대본 생성 (누가·언제·어떤 지시로 말하는지)
3. `verbalize(...)` — LLM(gpt-4o)이 각 턴의 한국어 문장만 작성
4. `assemble(...)` — 행동 파라미터에 따른 타임스탬프 부여

gold는 2단계 대본에서 결정되므로, 판정 LLM(gpt-4o-mini)과 생성 LLM(gpt-4o)의 분리와 함께 순환성을 이중 방어한다.

In [ ]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트

from eval.synth import LEVELS, SCENARIOS, build_spec, build_turn_script, realized_stats

# 대본 데모: LLM 없이 gold가 어떻게 행동 파라미터로 전개되는지 확인
spec = build_spec(l_dom=0.9, l_dep=0.1, scenario="daily")
script = build_turn_script(spec, seed=42)

print("gold:", spec["gold"])
print("behaviors:", spec["behaviors"])
print("realized:", realized_stats(script))
print("턴 수:", len(script))
script[:3]

## 3. 벤치마크 생성 (1회성, API 비용 발생)

5수준 × 4시나리오 × 5변형 = **100개 대화** (~1,500 메시지 × LLM 문장화).
gpt-4o 기준 대략 수 달러 수준. 아래 `RUN_GENERATION`을 `True`로 바꿔 명시적으로 실행한다.
이미 `data/synth/`에 생성본이 있으면 다시 실행할 필요 없다.

In [ ]:
import dataclasses
from llm.config import load_llm_config
from eval.synth import generate_dataset

RUN_GENERATION = False  # 비용 발생: 의도적으로만 True
GENERATOR_MODEL = "gpt-4o"  # 판정 모델(gpt-4o-mini)과 분리

if RUN_GENERATION:
  config = dataclasses.replace(load_llm_config(), model=GENERATOR_MODEL)
  result = generate_dataset("../data/synth", config,
                            progress=lambda n, cid: print(f"[{n:3d}/100] {cid}"))
  print("실패:", result["failed"] or "없음")
else:
  print("RUN_GENERATION=False — 기존 data/synth/ 를 사용합니다")

## 4. 데이터셋 EDA

In [ ]:
from pathlib import Path
import pandas as pd
from eval.datasets import load_synth_dir

SYNTH_DIR = Path("../data/synth")

if any(SYNTH_DIR.glob("*.json")):
  loaded = load_synth_dir(SYNTH_DIR)
  stats = pd.DataFrame([
    {
      "conversation_id": meta["conversation_id"],
      "scenario": meta["scenario"],
      "gold_dom": meta["gold"]["dominance"],
      "gold_dep": meta["gold"]["dependence"],
      "n_messages": len(df),
      "n_sessions": df["Session_ID"].nunique(),
      "emoticon_ratio": (df["Message_Type"] == "emoticon").mean(),
      "realized_initiation_a": meta["realized"]["initiation_ratio_a"],
      "realized_share_a": meta["realized"]["message_share_a"],
    }
    for meta, df in loaded
  ])
  display(stats.describe())
  display(stats.groupby("gold_dom")["realized_initiation_a"].mean())  # gold-실현 정합성
else:
  print("data/synth/ 가 비어 있습니다 — 3절의 생성 셀을 먼저 실행하세요")

In [ ]:
import plotly.express as px

if any(SYNTH_DIR.glob("*.json")):
  fig = px.scatter(stats, x="gold_dom", y="realized_initiation_a", color="scenario",
                   title="gold(dominance) vs 실현된 선톡 비율 — 대각선에 가까울수록 정합")
  fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash"))
  fig.show()

## 5. 공개 데이터: KOTE

감정 분류 벤치마크(NB02)용. 44라벨 → 6그룹 매핑은 `eval/label_maps.py` 참조.
로드 시 매핑-라벨 일치를 자체 검증한다 (어긋나면 즉시 실패).

In [ ]:
LOAD_KOTE = False  # 최초 1회 다운로드 (~50MB)

if LOAD_KOTE:
  from eval.datasets import load_kote
  kote = load_kote(split="test")
  print(kote)
  print("라벨:", kote.features["labels"].feature.names[:10], "...")

## 6. 산출물

- `data/synth/*.json` — gold label 포함 합성 벤치마크 100개
- `data/CARD.md` — 데이터 카드 (구성·라이선스·한계)
- 다음: **NB02 모델 선정 벤치마크** (KOTE 6그룹 projection에서 현행 KLUE-BERT vs 대안 모델 vs LLM)